# Voxa v4: Prototypical Architecture Calibration & Testing
This notebook implements the v4 architecture for Project Voxa. 
It replaces the MFCC+DTW pipeline with a **Deep Prototypical Engine** using YAMNet and Silero VAD.
This is designed to be run on Kaggle using custom uploaded `.wav` datasets.

### Pipeline:
1. **Silero VAD** detects speech.
2. **Contextual Padding** forces the window to exactly 1.44s (23,040 samples).
3. **YAMNet** extracts exactly `[2, 1024]` frame embeddings.
4. **Temporal Halving** concatenates the two frames into a 2048-D vector.
5. **Quality Control & Bifurcation** handles dysregulation and outliers.
6. **Prototypical Matcher** scores live audio against stored centroids using Cosine Similarity.

In [ ]:
!pip install -q torchaudio librosa scikit-learn

import os\nimport pandas as pd
import glob
import torch
import torchaudio
import librosa
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
from sklearn.cluster import KMeans
from scipy.spatial.distance import cosine

# Load YAMNet from TF Hub
print("Loading YAMNet...")
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')

# Load Silero VAD from PyTorch Hub
print("Loading Silero VAD...")
silero_vad, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad',
                                   model='silero_vad',
                                   force_reload=False)
(get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = utils

print("Models loaded successfully.")

## 1.5. Download ReCANVo Dataset from Zenodo

In [ ]:
import os
import urllib.request
import zipfile

dataset_url = "https://zenodo.org/records/5786860/files/ReCANVo.zip?download=1"
dataset_zip = "ReCANVo.zip"
extracted_dir = "ReCANVo_Dataset"

if not os.path.exists(extracted_dir):
    print("Downloading ReCANVo dataset from Zenodo (this may take a few minutes)...")
    urllib.request.urlretrieve(dataset_url, dataset_zip)
    print("Download complete. Extracting...")
    with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
        zip_ref.extractall(extracted_dir)
    print("Extraction complete.")
    os.remove(dataset_zip)
else:
    print("Dataset already downloaded and extracted.")

# Set the global dataset path for the pipeline
DATASET_DIR = os.path.join(os.getcwd(), extracted_dir)

## 2. DSP Pipeline (Padding & YAMNet)

In [ ]:
def apply_vad_and_padding(wav_path, target_sr=16000, pad_duration_sec=1.44):
    """
    Reads audio and contextually pads/crops it to exactly 1.44 seconds (23040 samples).
    BYPASSES VAD because ReCANVo audio clips are already pre-segmented.
    """
    # Load audio
    wav, sr = torchaudio.load(wav_path)
    
    # Convert stereo to mono if necessary
    if wav.shape[0] > 1:
        wav = torch.mean(wav, dim=0, keepdim=True)
        
    if sr != target_sr:
        wav = torchaudio.transforms.Resample(sr, target_sr)(wav)
        
    wav = wav.squeeze(0)
    audio_np = wav.numpy()
    
    # Simply take the center of the pre-segmented clip
    center = len(audio_np) // 2
    
    target_samples = int(pad_duration_sec * target_sr) # 1.44s * 16000 = 23040
    half_target = target_samples // 2
    
    start_idx = center - half_target
    end_idx = center + half_target
    
    out_audio = np.zeros(target_samples, dtype=np.float32)
    
    valid_start = max(0, start_idx)
    valid_end = min(len(audio_np), end_idx)
    
    out_start = max(0, -start_idx)
    out_end = out_start + (valid_end - valid_start)
    
    out_audio[out_start:out_end] = audio_np[valid_start:valid_end]
    
    return out_audio\n\ndef extract_2048d_vector(audio_16k_np):
    """
    Passes 1.44s audio into YAMNet to get [2, 1024] embeddings.
    Performs Temporal Halving by concatenating frame 0 and frame 1 to get a 2048-D vector.
    """
    # YAMNet requires inputs in range [-1.0, 1.0]
    scores, embeddings, spectrogram = yamnet_model(audio_16k_np)
    
    # embeddings shape should be exactly (2, 1024) because 1.44s audio provides 2 frames
    emb_np = embeddings.numpy()
    
    if emb_np.shape[0] < 2:
        # Fallback (shouldn't happen with strict 1.44s padding, but safe to have)
        padded = np.zeros((2, 1024), dtype=np.float32)
        padded[:emb_np.shape[0], :] = emb_np
        emb_np = padded
        
    # Temporal Halving: Concatenate First Half and Second Half
    # mathematically guaranteed by pad duration
    vector_2048 = np.concatenate([emb_np[0], emb_np[1]])
    
    # Normalize the final vector (helps with Cosine Distance stability)
    norm = np.linalg.norm(vector_2048)
    if norm > 0:
        vector_2048 = vector_2048 / norm
        
    return vector_2048

## 3. Enrollment & Quality Control

In [ ]:
def strict_outlier_rejection(vectors, max_sigma=2.0):
    """
    Discards vectors whose cosine distance to the initial centroid is > mu + max_sigma * std.
    """
    vectors = np.array(vectors)
    if len(vectors) < 3:
        return vectors
        
    initial_centroid = np.mean(vectors, axis=0)
    distances = [cosine(v, initial_centroid) for v in vectors]
    
    mu = np.mean(distances)
    std = np.std(distances)
    threshold = mu + (max_sigma * std)
    
    valid_vectors = [v for v, d in zip(vectors, distances) if d <= threshold]
    return np.array(valid_vectors)

def bifurcate_if_needed(valid_vectors, ood_threshold=0.3):
    """
    Handles intra-speaker variance (calm vs dysregulated).
    Requires minimum 12 samples to bifurcate.
    """
    num_samples = len(valid_vectors)
    if num_samples == 0:
        return []
        
    if num_samples < 12:
        print(f"Only {num_samples} valid samples. Forcing single centroid.")
        return [np.mean(valid_vectors, axis=0)]
        
    # Calculate max pairwise distance
    max_pairwise = 0.0
    for i in range(num_samples):
        for j in range(i+1, num_samples):
            dist = cosine(valid_vectors[i], valid_vectors[j])
            if dist > max_pairwise:
                max_pairwise = dist
                
    # Trigger threshold: if max pairwise > half the OOD ambient noise threshold
    trigger_threshold = ood_threshold * 0.5
    
    if max_pairwise > trigger_threshold:
        print(f"High variance detected (max_dist={max_pairwise:.4f} > {trigger_threshold:.4f}). Bifurcating via K-Means (K=2)...")
        kmeans = KMeans(n_clusters=2, n_init='auto', random_state=42).fit(valid_vectors)
        return [kmeans.cluster_centers_[0], kmeans.cluster_centers_[1]]
    else:
        print(f"Variance within limits (max_dist={max_pairwise:.4f}). Creating single centroid.")
        return [np.mean(valid_vectors, axis=0)]

def enroll_intent(wav_paths):
    """
    Full enrollment pipeline for a single intent.
    """
    vectors = []
    for path in wav_paths:
        audio = apply_vad_and_padding(path)
        if audio is not None:
            vec = extract_2048d_vector(audio)
            vectors.append(vec)
            
    if not vectors:
        return [], 0.82
        
    print(f"Extracted {len(vectors)} vectors before QC.")
    valid_vectors = strict_outlier_rejection(vectors)
    print(f"Retained {len(valid_vectors)} vectors after QC.")
    
    centroids = bifurcate_if_needed(valid_vectors)
    
    # Calculate dynamic OOD threshold based on child's acoustic variance
    if len(valid_vectors) > 0:
        similarities = []
        for v in valid_vectors:
            max_sim = max([1.0 - cosine(v, c) for c in centroids])
            similarities.append(max_sim)
        mean_sim = np.mean(similarities)
        std_sim = np.std(similarities)
        ood_threshold = max(0.82, mean_sim - 2 * std_sim)
    else:
        ood_threshold = 0.82
        
    return centroids, ood_threshold

## 4. Prototypical Matcher (Gates & Penalty)

In [ ]:
class PrototypicalMatcher:
    def __init__(self, ccp_penalty=0.005, margin_threshold=0.04):
        self.enrolled_intents = {} # { intent_name: {'centroids': [c1, c2], 'ood_threshold': 0.85} }
        self.ccp_penalty = ccp_penalty
        self.margin_threshold = margin_threshold
        
    def add_intent(self, intent_name, centroids, ood_threshold):
        self.enrolled_intents[intent_name] = {
            'centroids': centroids,
            'ood_threshold': ood_threshold
        }
        
    def classify(self, live_vector):
        if not self.enrolled_intents:
            return "Reject", "No intents enrolled"
            
        scores = {}
        thresholds = {}
        for intent_name, intent_data in self.enrolled_intents.items():
            centroids = intent_data['centroids']
            ood_threshold = intent_data['ood_threshold']
            # Calculate Cosine Similarity to each centroid
            similarities = [1.0 - cosine(live_vector, c) for c in centroids]
            max_sim = max(similarities)
            
            # Apply Centroid Count Penalty (CCP)
            k = len(centroids)
            effective_sim = max_sim - ((k - 1) * self.ccp_penalty)
            scores[intent_name] = effective_sim
            thresholds[intent_name] = ood_threshold
            
        # Sort by best effective similarity
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        best_intent, best_score = ranked[0]
        best_threshold = thresholds[best_intent]
        
        # OOD Gate Check
        if best_score < best_threshold:
            return "Reject_OOD", f"Best match ({best_intent}) score {best_score:.4f} is below OOD threshold {best_threshold:.4f}"
            
        # Margin Check (if more than 1 intent)
        if len(ranked) > 1:
            second_intent, second_score = ranked[1]
            margin = best_score - second_score
            if margin < self.margin_threshold:
                return "Reject_Margin", f"Ambiguous match. Margin {margin:.4f} below threshold {self.margin_threshold:.4f}. Best: {best_intent}, Second: {second_intent}"
                
        return "Accept", f"Match: {best_intent} (Score: {best_score:.4f})" 

## 5. Kaggle Dataset Loader & Testing

In [ ]:
import pandas as pd
import glob
import random
import os

def load_recanvo_dataset(dataset_dir):
    """
    Loads the ReCANVo dataset using the dataset_file_directory.csv.
    Finds the filenames and their corresponding affective/communicative labels.
    """
    csv_path = os.path.join(dataset_dir, 'dataset_file_directory.csv')
    intent_paths = {}
    
    if not os.path.exists(csv_path):
        # Fallback check if it's inside a ReCANVo subfolder
        csv_path = os.path.join(dataset_dir, 'ReCANVo', 'dataset_file_directory.csv')
        if not os.path.exists(csv_path):
            print(f"CSV not found in {dataset_dir}. Please ensure the ReCANVo dataset is extracted.")
            return intent_paths
            
    df = pd.read_csv(csv_path)
    
    # Find likely column names for filename and label
    filename_col = next((c for c in df.columns if 'file' in c.lower() or 'name' in c.lower()), df.columns[0])
    label_col = next((c for c in df.columns if 'label' in c.lower() or 'meaning' in c.lower() or 'affect' in c.lower()), df.columns[-1])
    
    print(f"Using column '{filename_col}' for filenames and '{label_col}' for labels.")
    
    for _, row in df.iterrows():
        fname = str(row[filename_col]).strip()
        if not fname.endswith('.wav'):
            fname += '.wav'
            
        label = str(row[label_col]).strip().lower()
        
        # Search for the file
        filepath = os.path.join(os.path.dirname(csv_path), fname)
        if not os.path.exists(filepath):
            # Attempt recursive search if it's not flat
            found = glob.glob(os.path.join(os.path.dirname(csv_path), "**", fname), recursive=True)
            if found:
                filepath = found[0]
            else:
                continue
                
        if label not in intent_paths:
            intent_paths[label] = []
        intent_paths[label].append(filepath)
        
    for label, paths in intent_paths.items():
        print(f"Found {len(paths)} samples for '{label}'")
        
    return intent_paths

In [ ]:
## 6. Execution & Testing Phases (ReCANVo Dataset)

# --- Configuration ---
# UPDATE THIS PATH to match your Kaggle dataset upload directory for ReCANVo
# DATASET_DIR is dynamically set in the Zenodo Download cell above 

# --- Initialization ---
intents_data = load_recanvo_dataset(DATASET_DIR)
matcher = PrototypicalMatcher()

if not intents_data:
    print("No data loaded. Check your DATASET_DIR path.")
else:
    # --- Split Data: 80% Enrollment, 20% Testing ---
    random.seed(42)
    enrollment_data = {}
    testing_data = {}

    for label, wavs in intents_data.items():
        # Only test on intents that have a reasonable amount of data (e.g. > 10 samples)
        if len(wavs) < 10:
            continue 
        
        random.shuffle(wavs)
        split_idx = int(len(wavs) * 0.8)
        enrollment_data[label] = wavs[:split_idx]
        testing_data[label] = wavs[split_idx:]

    # --- 1. Enrollment Phase ---
    print("\n=== STARTING ENROLLMENT ===")
    enrolled_count = 0
    for name, wavs in enrollment_data.items():
        print(f"\nEnrolling intent: {name} ({len(wavs)} files)")
        centroids, ood_threshold = enroll_intent(wavs)
        if centroids:
            matcher.add_intent(name, centroids, ood_threshold)
            enrolled_count += 1

    if enrolled_count == 0:
        print("\nERROR: No intents were enrolled.")
    else:
        print(f"\nEnrollment complete. {enrolled_count} intents active.")

        # --- 2. Testing Phase: Accuracy & FRR ---
        print("\n=== PHASE 2: TESTING RECANVO EVAL SET ===")
        
        correct = 0
        total = 0
        false_rejects = 0
        false_accepts = 0
        
        for true_label, wavs in testing_data.items():
            if true_label not in matcher.enrolled_intents:
                continue
                
            for wav in wavs:
                audio = apply_vad_and_padding(wav)
                if audio is not None:
                    vec = extract_2048d_vector(audio)
                    status, msg = matcher.classify(vec)
                    total += 1
                    
                    if status == "Accept":
                        # Parse the best intent from the message
                        predicted = msg.split("Match: ")[1].split(" (")[0]
                        if predicted == true_label:
                            correct += 1
                        else:
                            false_accepts += 1
                            # print(f"  [CONFUSION] True: {true_label}, Pred: {predicted}")
                    else:
                        false_rejects += 1
                        # print(f"  [REJECT] True: {true_label} -> {msg}")
                        
        if total > 0:
            acc = (correct / total) * 100
            frr = (false_rejects / total) * 100
            far = (false_accepts / total) * 100
            print(f"\n=== FINAL METRICS ===")
            print(f"Overall Accuracy: {acc:.2f}% ({correct}/{total} correct)")
            print(f"False Rejection Rate (FRR): {frr:.2f}% ({false_rejects}/{total} rejected)")
            print(f"Confusion/False Accept Rate: {far:.2f}% ({false_accepts}/{total} wrong intent)")

    print("\n=== TESTING COMPLETE ===")